In [ ]:
%pip install pandas

KPI 2 Score par Copro :

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_base_copro.csv')

df.columns = df.columns.str.strip().str.lower()

cols_numeriques = ['lots_habitation', 'lots_parking', 'total_lots']
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


df_propre = df[df['lots_habitation'] >= 5].copy()
df_propre['ratio_habitation'] = df_propre['lots_habitation'] / df_propre['total_lots'].replace(0, np.nan)
df_propre = df_propre[df_propre['ratio_habitation'] >= 0.3].copy()


df_unique = df_propre.groupby(['adresse', 'ville', 'code_postal', 'dept_code', 'dept_nom']).agg({
    'lots_habitation': 'max', 
    'lots_parking': 'max',     
    'lat': 'first',
    'long': 'first'
}).reset_index()


df_unique['score_immeuble'] = df_unique['lots_parking'] / df_unique['lots_habitation'].replace(0, 1)

df_final = df_unique[(df_unique['score_immeuble'] >= 0.5) & (df_unique['score_immeuble'] <= 0.9)].copy()

df_final['code_postal'] = df_final['code_postal'].fillna(0).astype(int).astype(str)


kpi2_score_immeuble = df_final[['code_postal', 'ville', 'adresse', 'lots_habitation', 'score_immeuble']].copy()

kpi2_score_immeuble['score_immeuble'] = kpi2_score_immeuble['score_immeuble'].round(2)

kpi2_score_immeuble = kpi2_score_immeuble.sort_values('lots_habitation', ascending=False)

kpi2_score_immeuble = kpi2_score_immeuble.drop_duplicates(subset=['code_postal', 'lots_habitation'], keep='first')

kpi2_score_immeuble = kpi2_score_immeuble.reset_index(drop=True)

display(kpi2_score_immeuble.head(10))

,code_postal,ville,adresse,lots_habitation,score_immeuble
0,78150,Le Chesnay,2 av charles de gaulle 78150 Le Chesnay,7527,0.83
1,78250,Meulan-en-Yvelines,22 all de la claire fontaine 78250 Meulan-en-Y...,2000,0.57
2,78170,La Celle-Saint-Cloud,18 Avenue de la Jonchere 78170 La Celle-Saint-...,1481,0.82
3,76600,Le Havre,12 r frederick lemaitre 76600 Le Havre,1072,0.88
4,34080,Montpellier,949 av du professeur louis ravas 34080 Montpel...,918,0.75
5,69100,VILLEURBANNE,"r du 1er mars 1943, 69100 Villeurbanne",864,0.71
6,78370,Plaisir,Rue des Ebisoires 78370 Plaisir,854,0.87
7,34090,Montpellier,250 rte de mende 34090 Montpellier,853,0.89
8,92160,Antony,24 Avenue Raymond Aron 92160 Antony,804,0.56
9,31200,Toulouse,31 av de mazades 31200 Toulouse,789,0.50


Export des données pour l'équipe Front-End:

In [ ]:
kpi2_score_immeuble.to_csv('export_kpi2_immeubles.csv', index=False, encoding='utf-8')